# 01 — PxWeb metadata and download (multi-table)

**Purpose.** For each table listed in `configs/data.yaml::tables_to_ingest`: fetch PxWeb metadata, POST one or more JSON-stat2 slices, save raw responses with SHA-256 provenance, and write a per-table manifest. 

- **Inputs:** `configs/data.yaml`, outbound to `https://pxdata.stat.fi/PxWeb/api/v1`.
- **Outputs (per table):**
  - `data/raw/<table_id>__meta.json` (table metadata)
  - `data/raw/<table_id>__data__slice_<n>.json` (one per slice)
  - `data/manifests/fetch_<table_id>_<timestamp>.json` (per-table provenance)
- **Aggregate outputs:**
  - `data/manifests/ingest_run_<timestamp>.json` (run summary across all tables)

Slicing is essential: KEHA tables have millions of cells in their full form and the PxWeb API rejects requests above ~500k cells with HTTP 403. Each table's `slices` block in `configs/data.yaml` declares the cell selections to use. The loop below also resolves the `last_24` and `tol_broad` sentinels against the actual table metadata.

In [ ]:
import os, json, hashlib, datetime as dt, time
from pathlib import Path
import requests, certifi

def _find_repo():
    env = os.environ.get("FINEYE_REPO")
    if env:
        return Path(env).resolve()
    p = Path.cwd().resolve()
    for cand in (p, *p.parents):
        if (cand / "configs" / "data.yaml").is_file():
            return cand
    return p
REPO = _find_repo()
RAW  = REPO / "data" / "raw"
MAN  = REPO / "data" / "manifests"
for d in (RAW, MAN):
    d.mkdir(parents=True, exist_ok=True)
CA_BUNDLE = certifi.where()
BASE = "https://pxdata.stat.fi/PxWeb/api/v1"
TIMEOUT = 120
NOW_UTC = dt.datetime.now(dt.timezone.utc)
TS = NOW_UTC.strftime("%Y%m%dT%H%M%SZ")
print("repo :", REPO)
print("base :", BASE)
print("ts   :", TS)

In [ ]:
try:
    import yaml
    CFG = yaml.safe_load(open(REPO / "configs" / "data.yaml"))
except Exception as e:
    raise SystemExit(f"Failed to load configs/data.yaml: {e}")
TABLES_CFG = CFG["tables"]
INGEST_LIST = CFG["tables_to_ingest"]
print("tables to ingest:", INGEST_LIST)
for k in INGEST_LIST:
    t = TABLES_CFG[k]
    print(f"  {k:50s} slices={len(t.get('slices', []))}  path={t['path']}")

In [ ]:
def http_get_json(url):
    try:
        r = requests.get(url, headers={"Accept": "application/json"}, timeout=TIMEOUT, verify=CA_BUNDLE)
        return {"ok": r.ok, "status": r.status_code, "body": r.content, "error": None if r.ok else r.text}
    except requests.RequestException as e:
        return {"ok": False, "status": None, "body": b"", "error": str(e)}

def http_post_json(url, payload):
    try:
        r = requests.post(
            url, json=payload,
            headers={"Content-Type": "application/json", "Accept": "application/json"},
            timeout=TIMEOUT, verify=CA_BUNDLE,
        )
        return {"ok": r.ok, "status": r.status_code, "body": r.content, "error": None if r.ok else r.text}
    except requests.RequestException as e:
        return {"ok": False, "status": None, "body": b"", "error": str(e)}

def sha256(b):
    return hashlib.sha256(b).hexdigest()

print("helpers ready")

In [ ]:
# Fetch and cache metadata for every table. One GET per table; small responses.
table_metas = {}
for key in INGEST_LIST:
    path = TABLES_CFG[key]["path"]
    table_id = path.split("/")[-1].replace(".px", "")
    meta_url = f"{BASE}/en/{path}"
    resp = http_get_json(meta_url)
    if not resp["ok"]:
        raise SystemExit(f"metadata fetch failed for {path}: status={resp['status']} error={resp['error']}")
    meta = json.loads(resp["body"].decode("utf-8"))
    raw_meta_path = RAW / f"{table_id}__meta.json"
    raw_meta_path.write_bytes(resp["body"])
    table_metas[key] = {"table_id": table_id, "path": path, "meta": meta, "raw_meta_path": raw_meta_path}
    print(f"  {key:50s} {table_id:8s} title={meta.get('title')!r:80s} sha={sha256(resp['body'])[:12]}")
print(f"\nfetched metadata for {len(table_metas)} tables")

In [ ]:
# Build a dimension-catalog: for every (table, dimension) the full set of codes.
# This is the single source of truth for sentinel resolution below.
dimension_catalog = {}
for key, info in table_metas.items():
    table_id = info["table_id"]
    dim_map = {}
    for v in info["meta"].get("variables", []):
        dim_map[v["code"]] = {
            "name": v.get("name"),
            "text": v.get("text"),
            "values": list(v.get("values") or []),
            "value_texts": list(v.get("valueTexts") or []),
        }
    dimension_catalog[table_id] = dim_map
cat_path = MAN / "_dimension_catalog.json"
cat_path.write_text(json.dumps(dimension_catalog, ensure_ascii=False, indent=1))
print("dimension catalog written:", cat_path)
for tid, dims in dimension_catalog.items():
    print(f"  {tid}: {', '.join(f'{c}({len(d["values"])})' for c, d in dims.items())}")

In [ ]:
# Resolve the configured query for each slice: replace sentinels with actual codes.
def resolve_query(table_id, declared_query, dims):
    resolved = {}
    for code, val in declared_query.items():
        if code not in dims:
            raise SystemExit(f"table {table_id}: query asks for dim {code!r} which is not in metadata dims {list(dims)}")
        if isinstance(val, str):
            if val == "all":
                resolved[code] = list(dims[code]["values"])
            elif val == "last_24":
                all_values = list(dims[code]["values"])
                # Take the last 24 monthly codes (YYYYMM sorts lexicographically)
                resolved[code] = [v for v in all_values if v >= "2024M01"]
            elif val == "tol_broad":
                # Top-level TOL industries: 1-letter codes A..U, excluding X/SSS/NO/Other
                resolved[code] = [v for v in dims[code]["values"] if len(v) <= 2 and v not in ("SSS", "X", "NO", "Other")]
            else:
                raise SystemExit(f"unknown sentinel {val!r} for dim {code!r} on table {table_id}")
        else:
            # Validate the codes exist in the table
            missing = [v for v in val if v not in dims[code]["values"]]
            if missing:
                raise SystemExit(f"table {table_id}: dim {code!r} has unknown values {missing[:5]} (showing first 5)")
            resolved[code] = list(val)
    return resolved

# Sanity check resolution for every table without hitting the API yet
for key in INGEST_LIST:
    table_id = table_metas[key]["table_id"]
    dims = dimension_catalog[table_id]
    for i, slc in enumerate(TABLES_CFG[key].get("slices", [])):
        resolved = resolve_query(table_id, slc["query"], dims)
        n_cells = 1
        for c, vs in resolved.items():
            n_cells *= len(vs)
        print(f"  {key:50s} slice[{i}] n_cells={n_cells:>9,d}  dims={list(resolved)}")

In [ ]:
# Fetch every slice. Track timing and HTTP status per slice.
fetch_log = []
for key in INGEST_LIST:
    table_id = table_metas[key]["table_id"]
    dims = dimension_catalog[table_id]
    data_url = f"{BASE}/en/{TABLES_CFG[key]['path']}"
    for i, slc in enumerate(TABLES_CFG[key].get("slices", [])):
        resolved = resolve_query(table_id, slc["query"], dims)
        payload = {
            "query": [{"code": c, "selection": {"filter": "item", "values": vs}} for c, vs in resolved.items()],
            "response": {"format": "JSON-STAT2"},
        }
        slice_path = RAW / f"{table_id}__data__slice_{i}.json"
        n_cells = 1
        for vs in resolved.values():
            n_cells *= len(vs)
        t0 = time.time()
        resp = http_post_json(data_url, payload)
        dt_s = time.time() - t0
        entry = {
            "table_key": key,
            "table_id": table_id,
            "slice_index": i,
            "slice_path": str(slice_path.relative_to(REPO)),
            "description": slc.get("description", ""),
            "n_cells_requested": n_cells,
            "http_status": resp["status"],
            "ok": resp["ok"],
            "bytes": len(resp["body"]),
            "elapsed_s": round(dt_s, 2),
        }
        if resp["ok"]:
            slice_path.write_bytes(resp["body"])
            entry["sha256"] = sha256(resp["body"])
            try:
                j = json.loads(resp["body"])
                entry["n_values_returned"] = len(j.get("value") or [])
                entry["size_declared"] = j.get("size")
                entry["source"] = j.get("source")
                entry["updated"] = j.get("updated")
            except Exception as e:
                entry["parse_error"] = str(e)
        else:
            entry["error"] = (resp.get("error") or b"")[:200]
        fetch_log.append(entry)
        status_str = f"{resp['status']} {len(resp['body']):>9,d}B {dt_s:5.1f}s"
        print(f"  {key:50s} slice[{i}] n={n_cells:>9,d}  {status_str}")
        if not resp["ok"]:
            print(f"    !! error: {entry.get('error', '')}")
print(f"\nfetch_log entries: {len(fetch_log)}")

In [ ]:
# Write per-table provenance manifests.
per_table_manifests = {}
for key in INGEST_LIST:
    table_id = table_metas[key]["table_id"]
    info = table_metas[key]
    meta = info["meta"]
    meta_path = info["raw_meta_path"]
    table_slices = [e for e in fetch_log if e["table_id"] == table_id]
    manifest = {
        "table_key": key,
        "table_id": table_id,
        "path": TABLES_CFG[key]["path"],
        "title": meta.get("title"),
        "fetched_at_utc": NOW_UTC.isoformat(timespec="seconds"),
        "pxweb_base_url": BASE,
        "response_format": "JSON-STAT2",
        "raw": {
            "meta_path": str(meta_path.relative_to(REPO)),
            "meta_sha256": sha256(meta_path.read_bytes()),
            "slices": [
                {
                    "slice_index": e["slice_index"],
                    "slice_path": e["slice_path"],
                    "sha256": e.get("sha256"),
                    "n_cells_requested": e["n_cells_requested"],
                    "n_values_returned": e.get("n_values_returned"),
                    "size_declared": e.get("size_declared"),
                    "bytes": e.get("bytes"),
                    "elapsed_s": e.get("elapsed_s"),
                    "http_status": e.get("http_status"),
                    "ok": e.get("ok"),
                    "description": e.get("description"),
                    "source": e.get("source"),
                    "updated": e.get("updated"),
                    "error": e.get("error"),
                }
                for e in table_slices
            ],
        },
        "variables": [
            {"code": v["code"], "name": v.get("name"), "text": v.get("text"), "n_values": len(v.get("values") or [])}
            for v in meta.get("variables", [])
        ],
    }
    man_path = MAN / f"fetch_{table_id}_{TS}.json"
    man_path.write_text(json.dumps(manifest, indent=1, ensure_ascii=False))
    per_table_manifests[key] = man_path
    n_ok = sum(1 for e in table_slices if e.get("ok"))
    print(f"  {key:50s} manifest={man_path.name}  ok_slices={n_ok}/{len(table_slices)}")

In [ ]:
# Aggregate run-level manifest.
total_bytes = sum(e.get("bytes") or 0 for e in fetch_log)
total_elapsed = sum(e.get("elapsed_s") or 0 for e in fetch_log)
run = {
    "timestamp_utc": NOW_UTC.isoformat(timespec="seconds"),
    "n_tables": len(INGEST_LIST),
    "n_slices": len(fetch_log),
    "n_slices_ok": sum(1 for e in fetch_log if e.get("ok")),
    "n_slices_failed": sum(1 for e in fetch_log if not e.get("ok")),
    "total_bytes": total_bytes,
    "total_elapsed_s": round(total_elapsed, 2),
    "per_table_manifests": {k: str(v.relative_to(REPO)) for k, v in per_table_manifests.items()},
    "fetch_log": fetch_log,
}
run_path = MAN / f"ingest_run_{TS}.json"
run_path.write_text(json.dumps(run, indent=1, ensure_ascii=False))
print(f"\ningest run: {run['n_slices_ok']}/{run['n_slices']} slices OK, {total_bytes:,} bytes, {total_elapsed:.1f}s total")
print("run manifest:", run_path)